# EyeGaze Colab1.1 Training

Upload folder `Colab1.1` to Google Drive and put `archive.zip` in `MyDrive` or inside `Colab1.1`.

This notebook trains the deployable model:

```text
best.dat
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Colab1.1')
ZIP_NAME = 'archive.zip'

known_paths = [
    Path('/content/drive/MyDrive') / ZIP_NAME,
    PROJECT_DIR / ZIP_NAME,
    Path('/content/drive/MyDrive/Datasets') / ZIP_NAME,
]
zip_path = next((p for p in known_paths if p.exists()), None)
if zip_path is None:
    matches = sorted(Path('/content/drive/MyDrive').rglob(ZIP_NAME))
    zip_path = matches[0] if matches else None
if zip_path is None:
    raise FileNotFoundError('Cannot find archive.zip in Google Drive')

GAZECAPTURE_ZIP = str(zip_path)
WORKDIR = '/content/eyegaze_work'
OUT_DIR = '/content/eyegaze_model'
DRIVE_OUT_DIR = '/content/drive/MyDrive/eyegaze_model'

print('Project:', PROJECT_DIR)
print('Archive:', GAZECAPTURE_ZIP)
print('Output:', OUT_DIR)
print('Drive backup:', DRIVE_OUT_DIR)

!pip install -q pandas numpy opencv-python-headless pillow torch


In [ ]:
!python "$PROJECT_DIR/train_colab.py" --zip "$GAZECAPTURE_ZIP" --workdir "$WORKDIR" --out "$OUT_DIR" --epochs 30 --batch-size 128 --image-size 64 --face-size 96 --num-workers 2


In [ ]:
from pathlib import Path
import shutil

out_dir = Path(OUT_DIR)
drive_out = Path(DRIVE_OUT_DIR)
drive_out.mkdir(parents=True, exist_ok=True)
for name in ['best.dat', 'last.dat', 'history.json', 'manifest.csv', 'eval_test.json']:
    src = out_dir / name
    if src.exists():
        shutil.copy2(src, drive_out / name)
print('Saved to:', drive_out)
for path in sorted(drive_out.iterdir()):
    print(path.name, path.stat().st_size)


In [ ]:
from google.colab import files
files.download(f'{OUT_DIR}/best.dat')
